|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>Detokenization<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: the streaming detokenizer<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B')
print('vocab', tokenizer.vocab_size)

Write the streaming detokenizer.

No GPU, no tensors, and no interesting arithmetic. This stage is boring and it
is the source of most user-visible bugs in real servers, which is a fair
trade for one afternoon.

# Exercise 1: emit the difference, not the token

A character can be split across two tokens, so decoding one token at a time
produces replacement characters. Decode the prefix instead and emit whatever
is new.

In [ ]:
class IncrementalDetokenizer:
  """Turn a stream of token ids into a stream of complete characters."""
  def __init__(self, tokenizer):
    self.tokenizer = tokenizer
    self.token_ids = []
    self.num_emitted = 0      # the characters that the client has already

  def push(self, token_id):
    """-> the NEW text that this token completes. It can be ''."""
    self.token_ids.append(token_id)
    # Decode the whole prefix, not only the token.
    text = 
    # A replacement character at the end means that the last character is
    # not complete. Send nothing, and wait for the next token.

    return 

  def flush(self):
    """The stream ended. Send the rest, complete or not."""
    # A reply can stop in the middle of a character. Without this method
    # that character disappears, and the fuzz test below finds the loss.
    

  def _emit_from(self, text):
    # Send only the text that is new.
    new_text = 
    self.num_emitted = 
    return new_text

def stream_text(tokenizer, token_ids):
  """Send token_ids through a new detokenizer. -> all the text that it sent."""
  detokenizer = IncrementalDetokenizer(tokenizer)
  pushed = ''.join(detokenizer.push(token_id) for token_id in token_ids)
  return pushed + detokenizer.flush()

token_ids = tokenizer('hello world', add_special_tokens=False).input_ids
detokenizer = IncrementalDetokenizer(tokenizer)
print([detokenizer.push(token_id) for token_id in token_ids])

# Exercise 2: prove it with a fuzz test

The contract is an equality. Test it that way.

In [ ]:
import random
# The contract is an equality: the streamed text == the batch-decoded text.
# Prove it on many random token sequences, not on examples.
rng = random.Random(0)
failures = 0
for _ in range(500):
  token_ids = [rng.randrange(tokenizer.vocab_size) for _ in range(rng.randint(1, 40))]
  
print(f'{failures} failures in 500 random sequences')

hard = ['\U0001F468\u200d\U0001F469\u200d\U0001F467\u200d\U0001F466',
        '\U0001F3F3\ufe0f\u200d\U0001F308', 'caf\u00e9', '\u65e5\u672c\u8a9e',
        '\uc548\ub155\ud558\uc138\uc694']
for text in hard:
  token_ids = tokenizer(text, add_special_tokens=False).input_ids
  matches = 
  print(f'{matches}  {text!r}')

# Exercise 3: stop strings that straddle

The model emits ` EN` then `D`. Neither token contains `END`.

In [ ]:
def held_length(held, stop):
  """The length of the longest end of `held` that is a start of `stop`.
  That text can still become the stop string, so do not emit it yet."""
  

def stream_until_stop(tokenizer, token_ids, stop):
  """Emit text until `stop` appears. Never emit a start of `stop`.
  -> (the emitted text, True if the stop string appeared).

  The stop string can cross a token boundary, so you cannot look at tokens.
  And you cannot emit early: 'EN' must not show before you know that it
  was the start of 'END'.
  """
  detokenizer = IncrementalDetokenizer(tokenizer)
  emitted, held = [], ''
  for token_id in token_ids:
    held += detokenizer.push(token_id)
    # Is the stop string in the text now? Emit the text before it, and stop.
    
    keep = held_length(held, stop)
    emitted.append(held[:len(held) - keep])
    held = held[len(held) - keep:]
  return ''.join(emitted) + held + detokenizer.flush(), False

cases = [('Answer: yes. END OF LINE', 'END'),
         ('nothing to stop for', 'END'),
         ('the ENDING is near', 'END')]
for text, stop in cases:
  token_ids = tokenizer(text, add_special_tokens=False).input_ids
  emitted, stopped = stream_until_stop(tokenizer, token_ids, stop)
  print(f'{text!r}\n  -> {emitted!r}  stopped={stopped}')

### Before you open the solution

1. Run the third case, `'the ENDING is near'`. Did it stop? Is that
   correct? Would a user agree?
2. Your detokenizer decodes the whole prefix every time, which is O(n)
   per token and O(n^2) for a reply. At what output length does that
   start to matter, and what would you keep instead of all the ids?
3. What happens if you emit eagerly and check for the stop string
   afterwards? Describe what the user sees.